# Per-level contrasts: one level against the pooled rest

The pair screens compare the levels of a categorical column by
their means. A level can be off in a way a mean cannot show:
one site whose readings spread three times as wide as every
other site's, with the same centre, reads as no association at
all. The per-level contrast scan states that level directly.
Nothing is declared: for every low-cardinality categorical
column and every numeric measure, each level is compared with
the pooled rest of the rows, and the claim carries the
measured medians and IQRs.

Four sites report a reading. Site `K7` runs high; site `WIDE`
has the same centre as the others and three times the spread.


In [1]:
import numpy as np
import polars as pl
import onceover

i = np.arange(400)
base = 20.0 + (i % 97) * 0.1 + ((i // 97) % 10) * 0.13
sites, readings = [], []
for site, values in [
    ("A", base),
    ("B", base),
    ("K7", base + 4.0),
    ("WIDE", 25.4 + (base - 25.4) * 3.0),
]:
    sites += [site] * 400
    readings += list(values)
gauges = pl.DataFrame({"site": sites, "reading": readings})

report = onceover.profile(gauges)
for finding in report.findings:
    if finding["type"] == "level_contrast":
        print(finding["claim"])
print()
print(report.coverage["level_contrasts"]["detail"])


reading for site = 'WIDE' is more variable than a typical other level: IQR 14.955 against 4.985 (3x), medians 23.735 against 26.19 pooled
reading for site = 'K7' sits above the pooled rest: median 28.845 against 24.69 (a shift of 4.155, 0.65 pooled IQRs, +17% against the pooled median); no other level of site has a median more than 0.38 pooled IQRs from its pooled rest

1 groupings (1 categorical columns, 0 calendar parts of the axis) x 1 numeric measures: 1 pairs examined, 4 levels tested (8 location and scale tests in the Benjamini-Hochberg family), 2 contrasts reported (cap 20)


## Reading the result

A location statement gives the level's median against the
rest's, the shift as a count of pooled IQRs, and — when the
rest's median is not zero — the percentage against it. A
scale statement gives the level's IQR against the median
IQR of the other levels — the typical level's spread, not
the pooled rest's, whose IQR inflates as a mixture whenever
one level is shifted — and their ratio. A level is
stated when the shift clears 0.5 pooled IQRs or the IQR ratio
clears 1.5 (or 2/3 in the narrow direction), and when a
rank-sum test on that form survives Benjamini–Hochberg
correction over the whole scan: every level × measure × test
is one enumerable family, so the correction is exact rather
than estimated. When exactly one level of a grouping is off,
the claim adds how far the most-shifted other level sits from
its own pooled rest — the "no other site differs" clause that
makes the one level the story. A level and its rest each need
at least 30 complete rows before either can be stated.

The scale form is the one the mean-based screens cannot
produce: `WIDE`'s median agrees with the rest's, so no
association statement fires, and the contrast is the only
place the report says the site is off.


In [2]:
j = np.arange(400)
level_base = 20.0 + 10.0 * ((j * 37) % 100) / 100.0 + 5.0 * ((j * 91) % 100) / 100.0
sites, readings = [], []
for site, values in [("A", level_base), ("B", level_base), ("CM", level_base * 2.54)]:
    sites += [site] * 400
    readings += list(values)
units = pl.DataFrame({"site": sites, "reading": readings})

report = onceover.profile(units)
for finding in report.findings:
    if finding["type"] == "level_contrast":
        print(finding["claim"])


reading for site = 'CM' sits at a near-constant multiple of the pooled rest: median 69.85 against 27.5, a quotient of 2.54, and the level's deciles track 2.54x the pooled rest's deciles (largest decile deviation 0.0%)


## The constant-multiple form

A level whose values are one fixed multiple of the pooled
distribution is a different fact from a level that is merely
high: the quotient is the actionable number, as with a unit
error (2.54 is centimetres against inches). The scan tests
the nine decile quotients of the level against the rest; when
they concentrate at one quotient at least 1.25× from 1, the
statement takes the multiple form and names the quotient,
instead of the location and scale forms the same rows would
also support. The form also requires the level's IQR ratio
to move with the quotient: a rescaling scales the spread by
the same factor, while a pure shift on a distribution
bounded away from zero concentrates its decile quotients
without moving the spread, and stays a location statement. The form requires the rest's deciles to sit on
one side of zero — a quotient is not defined against a
distribution that straddles zero.


In [3]:
import datetime as dt

def _noise(i):
    x = i.astype(np.uint64) * np.uint64(0x9E3779B97F4A7C15)
    x = x ^ (x >> np.uint64(29))
    x = x * np.uint64(0xBF58476D1CE4E5B9)
    x = x ^ (x >> np.uint64(32))
    return (x % np.uint64(10_000)).astype(np.float64) / 10_000.0

start = dt.date(2023, 1, 1)  # a Sunday
days = [start + dt.timedelta(days=d) for d in range(731)]
k = np.arange(731)
values = 10.0 + _noise(k) + _noise(k + 4000)
values = values + np.array([3.0 if d.weekday() == 5 else 0.0 for d in days])
daily = pl.DataFrame({"day": days, "reading": values})

report = onceover.profile(daily)
for finding in report.findings:
    if finding["type"] == "level_contrast":
        print(finding["claim"])


reading for day of week of day = 'Saturday' sits above the pooled rest: median 14.03215 against 10.9571 (a shift of 3.07505, 4.05 pooled IQRs, +28% against the pooled median); no other level of day of week of day has a median more than 0.34 pooled IQRs from its pooled rest


## Calendar parts

The frame's date axis contributes its own groupings to the
same scan: day of week, month of year, and hour of day (only
when the axis carries times of day, not bare dates). A
calendar level must span at least two distinct cycles — two
different weeks for a weekday, two different years for a
month, two different days for an hour — so a short extract
cannot state a weekend contrast it observed once. The claim
names the part and the axis it came from.

What stays silent matters as much. Levels below the row
floor, pairs whose pooled distribution has no spread, and
clean data — including the same values under shuffled level
labels — produce no per-level statements; the coverage entry
records the groupings and measures considered, the pairs
examined, the levels tested, and the size of the correction
family either way.
